# Problem Set 11: Project Genesis – The Scholar-Prime (Week 11)

Welcome to Week 11. In this assignment, you will build an academic research agent named **Scholar-Prime** that queries scientific databases to automatically find relevant research papers for simulation modeling. You will use the open-source **Science-Skills** repository developed by **Google DeepMind**.

## Objectives:
1. **Setting up the Science Skills** (Cloning and Environment Setup)
2. **Building the Literature Retrieval Agent (`agent.py`)**
3. **Automated Search & Downloader (Tool Binding)**
4. **Parameter Extraction & Verification**

Let's build!

## Exercise 1: Setting up the Science Skills

Clone the official Google DeepMind science-skills repository into your workspace, sync the environment dependencies via `uv`, and run a test query on the OpenAlex CLI to resolve a researcher's identity (e.g. "Geoffrey Hinton").

```bash
git clone https://github.com/google-deepmind/science-skills.git
cd science-skills/skills/literature-search-openalex
uv sync
uv run scripts/openalex_cli.py resolve authors "Geoffrey Hinton"
```

Execute the Python validation cell below to verify that the CLI utility exists and responds correctly.

In [ ]:
import subprocess
import os

# Check if science-skills folder is cloned
openalex_dir = os.path.join('science-skills', 'skills', 'literature-search-openalex')
if os.path.exists(openalex_dir):
    print("✔ science-skills repository found!")
    # Try running the help command for openalex_cli.py
    try:
        cmd = ["uv", "run", "scripts/openalex_cli.py", "--help"]
        # Run within the directory
        res = subprocess.run(cmd, cwd=openalex_dir, capture_output=True, text=True, shell=True)
        if res.returncode == 0:
            print("✔ openalex_cli.py helper script validated!")
            print(res.stdout[:200] + "...")
        else:
            print("✖ Error running openalex_cli.py:", res.stderr)
    except Exception as e:
        print("✖ Error running CLI:", str(e))
else:
    print("✖ science-skills directory not found. Please clone the repository first!")

## Exercise 2: Building the Literature Retrieval Agent

Define your `scholar_prime` agent inside `agent.py` using the ADK SDK.

Complete the starter template below to match your configured `agent.py`.

In [ ]:
# Paste your cognitive_core/agent.py code here for submission
from google.adk.agents.llm_agent import Agent

# Define your Scholar-Prime Agent here
scholar_prime = Agent(
    model='gemini-3.5-flash',
    name='scholar_prime',
    description='An academic research agent specialized in querying scientific databases and extracting material parameters.',
    instruction='Du bist ein wissenschaftlicher Bibliotheks-Agent. Finde passende Publikationen, bewerte deren Relevanz und extrahiere Parameter.'
)

## Exercise 3: Automated Search & Downloader (Tool Binding)

Implement a Python function `search_arxiv(query: str, max_results: int = 5) -> str` that invokes the arXiv CLI tool from the science-skills repository under the hood. Bind this function to your agent and test it in the Web UI.

Below is the template for the wrapper tool.

In [ ]:
import subprocess
import os

def search_arxiv(query: str, max_results: int = 3) -> str:
    """
    Queries arXiv for research papers on a given topic.
    Args:
        query: The search term to query on arXiv.
        max_results: The maximum number of papers to return.
    Returns:
        A string summarizing the matching papers (title, authors, summary, DOI).
    """
    arxiv_dir = os.path.join('science-skills', 'skills', 'literature-search-arxiv')
    if not os.path.exists(arxiv_dir):
        return "Error: literature-search-arxiv folder not found."
    
    # Implement calling the arXiv CLI script using subprocess
    # Example command format:
    # uv run scripts/arxiv_cli.py filter works --search "<query>" --per-page <max_results>
    try:
        cmd = ["uv", "run", "scripts/arxiv_cli.py", "filter", "works", "--search", query, "--per-page", str(max_results)]
        res = subprocess.run(cmd, cwd=arxiv_dir, capture_output=True, text=True, shell=True)
        if res.returncode == 0:
            return res.stdout
        else:
            return f"CLI Error: {res.stderr}"
    except Exception as e:
        return f"Execution Error: {str(e)}"

# Test the tool locally
# print(search_arxiv("thermodynamic simulation parameters"))

## Exercise 4: Parameter Extraction & Verification

Define a structured parameter extraction tool. Write a python script that runs the full pipeline: searches arXiv, reads the top paper abstract, calls the extractor, and outputs a JSON file.

In [ ]:
import json

def extract_parameters_from_text(text: str) -> dict:
    """
    Extracts thermodynamic and material parameters from a scientific text.
    Args:
        text: The text snippet containing parameters (e.g. abstract).
    Returns:
        A dictionary of the extracted parameters.
    """
    # Put your extraction reasoning logic here or call a structured Gemini API schema
    # For starter, return a stub
    return {
        "material": "Uranium Dioxide",
        "thermal_conductivity_W_mK": 3.5,
        "melting_point_K": 3120.0,
        "density_g_cm3": 10.97,
        "extracted_from_text_preview": text[:100]
    }

# Mock run of the pipeline
mock_text = "This paper studies Uranium Dioxide (UO2) fuel material which has a melting point of 3120K, density of 10.97 g/cm3 and a baseline thermal conductivity of 3.5 W/(m*K)."
extracted = extract_parameters_from_text(mock_text)

# Save parameters
os.makedirs('docs', exist_ok=True)
with open('docs/simulation_parameters.json', 'w') as f:
    json.dump(extracted, f, indent=4)
print("✔ Saved parameters to docs/simulation_parameters.json!")